# 201 — Prepare Sentinel-2 Data

Downloads S2 10km tiles that intersect lupine polygons,
rasterizes lupine masks to 10m grid, QA checks, saves images and masks.

**Requires:** `config_s2.py`, `helpers_s2.py`, `helpers.py` in same folder.
Run before `202_train_sat.ipynb`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config_s2 import *
from helpers_s2 import *
import time
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"BASE_DIR: {BASE_DIR}")
print(f"Cores:    {N_CORES}")
check_required_files_s2()

In [ ]:
# ── Step 1: build tile list — tiles intersecting lupine ───────────────────────
print("── Step 1: Build tile list ──────────────────────────────────")

grid       = gpd.read_file(S2_GRID_PATH)
lupine_gdf = gpd.read_file(LUPINE_VEC, layer=LUPINE_LAYER)
lupine_gdf["geometry"] = lupine_gdf.geometry.make_valid()
lupine_gdf = lupine_gdf.to_crs(grid.crs)

lupine_union = lupine_gdf.union_all()
work_grid    = grid[grid.intersects(lupine_union)].copy()

url_map    = {row["full_name"]: row["url"] for _, row in work_grid.iterrows()}
all_fnames = sorted(url_map.keys())

print(f"  Total S2 tiles:            {len(grid)}")
print(f"  Tiles intersecting lupine: {len(work_grid)}")

# Resume
existing_masks = set(os.listdir(S2_MASK_DIR))
remaining      = [f for f in all_fnames if f not in existing_masks]
print(f"  Done:      {len(all_fnames) - len(remaining)}")
print(f"  Remaining: {len(remaining)}")

In [ ]:
# ── Step 2: download → rasterize mask → save image + mask ────────────────────
print("── Step 2: Download and create masks ────────────────────────")

qa_results = []
skipped    = 0
t0         = time.time()

for i, fname in enumerate(remaining):
    url        = url_map[fname]
    local_path = os.path.join(TMP_DIR, fname)
    remaining_count = len(remaining) - i - 1

    # Download
    if not os.path.exists(local_path):
        ok = safe_download(url, local_path)
        if not ok:
            print(f"  [!] Download failed — skipping {fname}")
            skipped += 1
            continue

    # Read
    try:
        with rasterio.open(local_path) as src:
            img            = src.read()
            tile_crs       = src.crs
            tile_transform = src.transform
            tile_shape     = (src.height, src.width)
            tile_bounds    = src.bounds
            tile_profile   = src.profile.copy()
    except Exception as e:
        print(f"  [!] Read failed {fname}: {e}")
        skipped += 1
        try: os.remove(local_path)
        except: pass
        continue

    # Rasterize lupine mask
    tile_box  = box(*tile_bounds)
    lup_clip  = lupine_gdf[lupine_gdf.to_crs(tile_crs).intersects(tile_box)].copy()
    lup_clip  = lup_clip.to_crs(tile_crs)

    mask = rasterize_vector(lup_clip, {
        "height": tile_shape[0], "width": tile_shape[1],
        "transform": tile_transform, "crs": tile_crs
    }) if len(lup_clip) > 0 else np.zeros(tile_shape, dtype=np.uint8)

    lupine_px = int((mask == 1).sum())
    nodata_px = int((img[0] >= NODATA_VAL).sum())
    total_px  = mask.size

    # Save image tile (keep as int16)
    img_profile = tile_profile.copy()
    img_profile.update(compress="lzw", tiled=True,
                       blockxsize=256, blockysize=256)
    with rasterio.open(os.path.join(S2_IMG_DIR, fname), "w", **img_profile) as dst:
        dst.write(img)

    # Save mask tile
    mask_profile = tile_profile.copy()
    mask_profile.update(count=1, dtype="uint8", compress="lzw",
                        tiled=True, blockxsize=256, blockysize=256,
                        nodata=None)
    with rasterio.open(os.path.join(S2_MASK_DIR, fname), "w", **mask_profile) as dst:
        dst.write(mask.astype(np.uint8)[np.newaxis, :, :])

    qa_results.append({
        "tile":       fname,
        "lupine_px":  lupine_px,
        "total_px":   total_px,
        "nodata_px":  nodata_px,
        "coverage":   round(lupine_px / total_px, 4),
        "nodata_pct": round(nodata_px / total_px, 4),
    })

    # Delete source tile
    if DELETE_AFTER:
        try: os.remove(local_path)
        except: pass

    del img, mask, lup_clip; gc.collect()

    elapsed = time.time() - t0
    rate    = (i + 1 - skipped) / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(remaining):
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────")
        print(f"     Elapsed: {elapsed/60:.1f}min  |  ETA: {eta/60:.1f}min\n")
    else:
        print(f"  [{i+1}/{len(remaining)}] {fname}  lupine={lupine_px:,}px  ETA {eta/60:.1f}min")

print(f"\nDone. Skipped: {skipped}  Time: {(time.time()-t0)/60:.1f}min")

In [ ]:
# ── Step 3: QA summary ────────────────────────────────────────────────────────
print("── Step 3: QA summary ───────────────────────────────────────")

qa_path = os.path.join(S2_DIR, "s2_mask_qa.csv")
if os.path.exists(qa_path) and not qa_results:
    qa_df = pd.read_csv(qa_path)
else:
    qa_df = pd.DataFrame(qa_results)
    qa_df.to_csv(qa_path, index=False)

print(f"  Total tiles:        {len(qa_df)}")
print(f"  With lupine:        {(qa_df['lupine_px'] > 0).sum()}")
print(f"  Mean coverage:      {qa_df['coverage'].mean():.2%}")
print(f"  High nodata (>20%): {(qa_df['nodata_pct'] > 0.2).sum()} tiles")
print(f"  QA saved → {qa_path}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
qa_df[qa_df["lupine_px"] > 0]["coverage"].hist(bins=40, ax=axes[0], color="green", alpha=0.8)
axes[0].set_title("Lupine coverage per tile")
axes[0].set_xlabel("Coverage")
qa_df["nodata_pct"].hist(bins=40, ax=axes[1], color="gray", alpha=0.8)
axes[1].set_title("Nodata % per tile")
axes[1].set_xlabel("Nodata fraction")
plt.tight_layout()
plt.savefig(os.path.join(S2_DIR, "qa_coverage.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Step 4: NDVI distribution ─────────────────────────────────────────────────
print("── Step 4: NDVI distribution ────────────────────────────────")

tiles_with_lupine = qa_df[qa_df["lupine_px"] > 0]["tile"].tolist()
all_ndvi_lupine   = []
all_ndvi_bg       = []

for fname in tiles_with_lupine[:20]:
    img_path  = os.path.join(S2_IMG_DIR, fname)
    mask_path = os.path.join(S2_MASK_DIR, fname)
    if not os.path.exists(img_path): continue

    with rasterio.open(img_path)  as src: img  = src.read().astype(np.float32)
    with rasterio.open(mask_path) as src: mask = src.read(1)

    valid = (img[0] < NODATA_VAL) & (img[0] > 0)
    B8    = np.where(valid, img[S2_BANDS.index("B8")]  / S2_SCALE, np.nan)
    B4    = np.where(valid, img[S2_BANDS.index("B4")]  / S2_SCALE, np.nan)
    NDVI  = (B8 - B4) / (B8 + B4 + 1e-6)

    lupine_mask = (mask == 1) & valid
    bg_mask     = (mask == 0) & valid

    if lupine_mask.sum() > 0:
        all_ndvi_lupine.extend(NDVI[lupine_mask].tolist())
    if bg_mask.sum() > 0:
        all_ndvi_bg.extend(NDVI[bg_mask].ravel()[:2000].tolist())

if all_ndvi_lupine:
    import numpy as np
    print(f"  Lupine NDVI — mean={np.nanmean(all_ndvi_lupine):.3f}  "
          f"p5={np.nanpercentile(all_ndvi_lupine,5):.3f}  "
          f"p95={np.nanpercentile(all_ndvi_lupine,95):.3f}")

    plt.figure(figsize=(10, 4))
    plt.hist([x for x in all_ndvi_lupine if not np.isnan(x)], bins=60, alpha=0.7, color="green", label="Lupine")
    plt.hist([x for x in all_ndvi_bg     if not np.isnan(x)], bins=60, alpha=0.4, color="gray",  label="Background")
    plt.xlabel("NDVI"); plt.ylabel("Pixel count")
    plt.title("NDVI distribution — S2 lupine vs background")
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(os.path.join(S2_DIR, "ndvi_distribution_s2.png"), dpi=150, bbox_inches="tight")
    plt.show()